# Human-in-the-Loop: All Edit Options

This notebook demonstrates 4 ways to fix the "agent retries original task after an edit" problem.

| Option | Strategy | How it works |
|--------|----------|--------------|
| 1 | Inject explanation message | Tell the agent the edit is done |
| 2 | Reject + new user message | Clear slate, fresh intent |
| 3 | Edit user message too | Align agent's goal with what was sent |
| 4 | End loop after edit | Don't let the agent continue at all |

## Shared Setup (run this first)

In [ ]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
from typing_extensions import TypedDict
from dotenv import load_dotenv
import uuid

load_dotenv()

USER = "Leo"


@tool()
def send_email(recipient: str, subject: str, body: str) -> str:
    """Send an email to a recipient.
    Args:
        recipient: Email address of the recipient.
        subject: Subject line of the email.
        body: Body content of the email.
    Returns:
        Confirmation message.
    """
    print(f"  >>> TOOL EXECUTED: send_email to {recipient} | subject: {subject}")
    return f"Email sent successfully to {recipient}"


def make_agent():
    """Create a fresh agent + config for each option."""
    agent = create_agent(
        model="openai:gpt-4o",
        tools=[send_email],
        system_prompt=(
            f"You are a helpful assistant for {USER} that can send emails. "
            "Always return a message about the content of a tool call."
        ),
        middleware=[HumanInTheLoopMiddleware(interrupt_on={"send_email": True})],
        checkpointer=InMemorySaver(),
    )
    config = {"configurable": {"thread_id": str(uuid.uuid4())}}
    return agent, config


# The edited email the human wants to send instead
EDITED_ACTION = {
    "name": "send_email",
    "args": {
        "recipient": "partner@startup.com",
        "subject": "Budget proposal for Q1 2026",
        "body": "I can only approve up to 500k, please send over details.",
    },
}

USER_INPUT = "can you send an email to leo@gmail.com asking for a meeting tomorrow at 10am?"


def run_initial_invoke(agent, config):
    """Run the first invoke and return result. Prints interrupt info."""
    result = agent.invoke(
        {"messages": [{"role": "user", "content": USER_INPUT}]},
        config=config,
    )
    if "__interrupt__" in result:
        info = result["__interrupt__"][-1].value["action_requests"][-1]
        print("[INTERRUPT] Agent wants to call:")
        print(f"  Tool     : {info['name']}")
        print(f"  recipient: {info['args']['recipient']}")
        print(f"  subject  : {info['args']['subject']}")
    return result


def print_final_messages(result):
    """Print just the last few messages so output is readable."""
    messages = result.get("messages", [])
    print(f"\n[FINAL STATE] {len(messages)} messages in thread:")
    for msg in messages[-4:]:  # show last 4
        role = type(msg).__name__
        content = getattr(msg, "content", "") or ""
        tool_calls = getattr(msg, "tool_calls", [])
        if tool_calls:
            for tc in tool_calls:
                print(f"  [{role}] → tool call: {tc['name']}({tc['args']})")
        elif content:
            print(f"  [{role}] {str(content)[:120]}")


print("Setup complete. Run any option cell below.")

---
## Option 1 — Inject an Explanation Message

**Strategy:** After executing the edited tool call, resume once more with an injected
`tool` message that explicitly tells the agent:
> *"Human edited this call. Original task is now complete. Stop."*

The agent reads this in its history and understands there is nothing left to do.

**Best when:** You want the agent to stay in the loop (e.g. multi-step workflows) but just
need it to know the edit counts as task completion.

In [ ]:
print("=" * 60)
print("OPTION 1 — Inject explanation message after edit")
print("=" * 60)

agent1, config1 = make_agent()

# Step 1: initial invoke → interrupted
result = run_initial_invoke(agent1, config1)

if "__interrupt__" in result:
    print("\n[HUMAN] Choosing EDIT — sending a completely different email instead.")

    # Step 2: Resume with the edit decision (executes the edited tool call)
    result = agent1.invoke(
        Command(resume={"decisions": [{"type": "edit", "edited_action": EDITED_ACTION}]}),
        config=config1,
    )

    # Step 3: If the agent is interrupted again (it wants to retry the original),
    # inject a system-level tool message that tells it the task is done.
    if "__interrupt__" in result:
        print("[AGENT] Tried to send another email — injecting completion message...")

        result = agent1.invoke(
            Command(
                resume={
                    "decisions": [
                        {
                            # We approve the second attempt but override its tool result
                            # with an explanation so the agent stops.
                            "type": "approve"
                        }
                    ]
                },
                # Inject a synthetic tool result that explains what happened
                update={
                    "messages": [
                        {
                            "role": "tool",
                            "content": (
                                "[SYSTEM] The human edited and executed the email in a previous step. "
                                "The original user request is now considered complete. "
                                "Do not send any additional emails."
                            ),
                            "tool_call_id": result["__interrupt__"][-1]
                                .value["action_requests"][-1].get("id", "injected"),
                        }
                    ]
                },
            ),
            config=config1,
        )

print_final_messages(result)
print("\n[RESULT] Agent stopped after edited email. No duplicate send.")

---
## Option 2 — Reject + New User Message

**Strategy:** Reject the original tool call entirely, which cancels it cleanly. Then send
a *new* user message that describes exactly what to send. The agent plans fresh from a
clean slate — no confusion about a mismatched tool result.

**Best when:** The edit is significant enough that the original intent no longer applies at
all. Gives the agent full context about the new goal.

In [ ]:
print("=" * 60)
print("OPTION 2 — Reject + send a corrected user message")
print("=" * 60)

agent2, config2 = make_agent()

# Step 1: initial invoke → interrupted
result = run_initial_invoke(agent2, config2)

if "__interrupt__" in result:
    print("\n[HUMAN] Choosing REJECT — will re-issue a corrected request.")

    # Step 2: Reject the original tool call entirely
    result = agent2.invoke(
        Command(
            resume={
                "decisions": [
                    {
                        "type": "reject",
                        "message": "Do not send this email.",
                    }
                ]
            }
        ),
        config=config2,
    )

    # Step 3: Send a new, corrected user message that describes exactly what to send.
    # The agent now has full context and will plan the correct email from scratch.
    corrected_request = (
        "Please send an email to partner@startup.com with subject "
        "'Budget proposal for Q1 2026' and body "
        "'I can only approve up to 500k, please send over details.'"
    )

    result = agent2.invoke(
        {"messages": [{"role": "user", "content": corrected_request}]},
        config=config2,
    )

    # This new request will also hit the interrupt — approve it
    if "__interrupt__" in result:
        print("[INTERRUPT] Agent prepared the corrected email — approving...")
        result = agent2.invoke(
            Command(resume={"decisions": [{"type": "approve"}]}),
            config=config2,
        )

print_final_messages(result)
print("\n[RESULT] Only the corrected email was sent. No retry of original.")

---
## Option 3 — Edit the User Message Too

**Strategy:** When applying the edit, also patch the *original user message* in the thread
state so the agent's goal aligns with what was actually sent. The agent sees:
> User asked for X → tool sent X → done.

No mismatch, no retry.

**Best when:** You want the conversation history to be coherent and self-consistent for
later inspection or multi-turn follow-ups.

In [ ]:
print("=" * 60)
print("OPTION 3 — Edit user message to match the edited tool call")
print("=" * 60)

agent3, config3 = make_agent()

# Step 1: initial invoke → interrupted
result = run_initial_invoke(agent3, config3)

if "__interrupt__" in result:
    print("\n[HUMAN] Choosing EDIT — also rewriting the user message to match.")

    # Build the rewritten user message that matches the edited email
    rewritten_user_message = (
        "Please send an email to partner@startup.com about the "
        "Budget proposal for Q1 2026, saying I can only approve up to 500k."
    )

    # Step 2: Resume with edit + simultaneously patch the user message in state
    # so the agent's goal matches what was sent.
    result = agent3.invoke(
        Command(
            resume={
                "decisions": [{"type": "edit", "edited_action": EDITED_ACTION}]
            },
            # Overwrite the first (user) message so the agent's goal matches reality
            update={
                "messages": [
                    {
                        "role": "user",
                        "content": rewritten_user_message,
                        # Use the same message id as the original to overwrite it
                        "id": result["messages"][0].id,
                    }
                ]
            },
        ),
        config=config3,
    )

print_final_messages(result)
print("\n[RESULT] User message and tool call are aligned — agent sees task as complete.")

---
## Option 4 — End the Agent Loop After Edit

**Strategy:** After the edited tool call executes successfully, **stop the agent loop
entirely** — don't give it a chance to continue reasoning. You manually extract the tool
result and present it to the user yourself.

**Best when:** This is the simplest and most reliable fix. The agent's opinion after an
edit is irrelevant — you control the output directly. Good for fire-and-forget actions
like sending emails, making API calls, etc.

In [ ]:
print("=" * 60)
print("OPTION 4 — End the loop after edit (simplest & most reliable)")
print("=" * 60)

agent4, config4 = make_agent()

# Step 1: initial invoke → interrupted
result = run_initial_invoke(agent4, config4)

if "__interrupt__" in result:
    print("\n[HUMAN] Choosing EDIT — will stop the loop immediately after execution.")

    # Step 2: Resume with the edit. The edited tool call executes.
    result = agent4.invoke(
        Command(resume={"decisions": [{"type": "edit", "edited_action": EDITED_ACTION}]}),
        config=config4,
    )

    # Step 3: STOP HERE — don't let the agent continue reasoning.
    # Find the tool result from the messages and surface it directly.
    tool_results = [
        msg for msg in result.get("messages", [])
        if getattr(msg, "type", "") == "tool"
        or type(msg).__name__ == "ToolMessage"
    ]

    if tool_results:
        last_tool_result = tool_results[-1]
        print(f"\n[DONE] Tool result: {last_tool_result.content}")
        print("[DONE] Agent loop terminated. No further reasoning allowed.")
    else:
        # The agent was interrupted again (wants to retry) — we simply ignore it
        print("\n[DONE] Edit was applied. Discarding agent's retry attempt.")
        print(f"[DONE] Edited email was sent to: {EDITED_ACTION['args']['recipient']}")

    # Do NOT call agent.invoke() again — the loop ends here.
    print("\n[RESULT] Exactly one email sent. Agent had no opportunity to retry.")

---
## Which Option Should You Use?

| Option | Complexity | Agent continues? | History coherent? | Best for |
|--------|------------|-----------------|-------------------|----------|
| 1 — Inject message | Medium | Yes, but told to stop | Mostly | Multi-step workflows |
| 2 — Reject + new message | Medium | Yes, fresh start | Yes | Significant intent change |
| 3 — Edit user message | Low | Yes, naturally stops | Yes ✓ | Clean audit trails |
| 4 — End loop | **Lowest** | No | Partial | Simple fire-and-forget actions |

> **Recommended default: Option 4** for simple tools like `send_email`.  
> **Recommended for complex workflows: Option 3** — keeps history consistent with zero extra round-trips.